In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations
from pandas.api.types import is_integer_dtype
from collections import Counter, defaultdict
from math import ceil

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
# Define the path to the processed data based on the current user
current_user = os.getlogin()

# Define output directory (notebook is in ./notebooks; data is ../data)
DATA_DIR     = Path(rf"/home/{current_user}/local-share")
OUT_DIR      = DATA_DIR / "processed"

low_frequency_path = OUT_DIR / "handled_low_frequency_categories.csv"

# Let pandas detect the delimiter
data = pd.read_csv(low_frequency_path, sep=None, engine="python", encoding="utf-8-sig")
print(data.shape)
data



In [ ]:
data.columns

In [ ]:
data.info()

In [ ]:
# NaN count per column in data
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

# Missing Values Imputation Strategy

In [ ]:
# -----------------------------------------------------------------------------
# Missing Value Imputation Plan — with Missingness Overview
#
# Context:
#   Dataset includes numeric, categorical, and date-like fields (Datum columns).
#
# Dataset size:
#   N = 47,953 rows
#
# Missingness Summary (by column group)
#
#   Orientation participation (numeric / date identifiers):
#     • sdo2_orientation_Last_Event_Date ...................... 38,727  (80.76%)
#     • sdo2_orientation_Number_of_Event_Types ................ 38,727  (80.76%)
#     • sdo2_orientation_Number_of_Events_Attended ............ 38,727  (80.76%)
#     • sdo2_orientation_STUDENTNUMMER ........................ 38,727  (80.76%)
#     • sdo2_orientation_First_Event_Date ..................... 38,727  (80.76%)
#
#   Academic performance (numeric):
#     • sdo6_results_Average_Grade_B .......................... 11,543  (24.07%)
#     • sdo6_results_Average_Grade_A ..........................  7,592  (15.83%)
#     • sdo6_results_Percentage_Credits_B .....................  6,428  (13.40%)
#     • sdo6_results_Potential_Credits_B ......................  6,028  (12.57%)
#     • sdo6_results_Percentage_Credits_A .....................  4,319  ( 9.01%)
#     • sdo6_results_Potential_Credits_A ......................  4,158  ( 8.67%)
#
#   SKC and event-related dates:
#     • sdo2_skc_SKC_DATUM ....................................  5,248  (10.94%)
#
#   Distances & postcodes (numeric):
#     • sdo1_prev_distance_postcode ...........................  4,538  ( 9.46%)
#     • sdo1_prev_distance_distance_to_3584_CS ................  4,538  ( 9.46%)
#     • sdo1_prev_distance_distance_to_3812_PA ................  4,538  ( 9.46%)
#     • sdo1_postal_distance_postcode .........................  1,536  ( 3.20%)
#     • sdo1_postal_distance_distance_to_3812_PA ..............  1,536  ( 3.20%)
#     • sdo1_postal_distance_distance_to_3584_CS ..............  1,536  ( 3.20%)
#
#   Demographics (numeric/categorical):
#     • sdo1_characteristics_Dutch_nationality ................  3,476  ( 7.25%)
#     • sdo1_characteristics_age_start_study ..................  3,476  ( 7.25%)
#
#   Previous education (date):
#     • sdo1_previous_Final_Exam_Date .........................    779  ( 1.62%)
#
#   Degree-level and indicator columns: 0% missing
#     • sdo5_degree_* fields
#     • has_orientation, has_previous, has_characteristics, has_skc, has_dsa, has_nf
#
# -----------------------------------------------------------------------------
# Imputation Plan
#
# 1) Orientation participation (≈81% missing — structural absence):
#      → Do not impute dates (keep as NaT).
#      → Numeric counts → 0.
#      → Structural missingness handled via has_orientation flag.
#
# 2) Academic performance (≈8–24% missing):
#      → Numeric columns → median imputation.
#      → Add <col>_missing_flag for interpretability.
#
# 3) Distances & postcodes (≈3–9% missing):
#      → Numeric columns → median imputation.
#      → Add <col>_missing_flag.
#
# 4) Demographics (≈7% missing):
#      → Age → median; Nationality → mode.
#      → Add missing flags.
#
# 5) Dates (partial missingness):
#      → Convert to datetime (errors='coerce').
#      → For columns like sdo2_skc_SKC_DATUM and sdo1_previous_Final_Exam_Date:
#            - Impute with median date (preserves chronological order).
#      → For structurally missing dates (orientation event dates):
#            - Keep NaT (represents “no event recorded”).
#
# 6) Columns with no missing values:
#      → No action required.
#
# -----------------------------------------------------------------------------
# Summary:
#   - Structural missingness (no participation) retained as NaT or 0.
#   - Random numeric gaps filled with medians; categorical with modes.
#   - Partial date gaps imputed with median date for temporal continuity.
#   - All imputations preserve feature meaning and chronological relationships.
# -----------------------------------------------------------------------------


In [ ]:
# =============================================================================
# PURPOSE
# -----------------------------------------------------------------------------
# Function: impute_all(data)
#
# This function performs structured missing value imputation for the full
# student dataset. It handles numeric, categorical, and date-like columns
# while preserving the meaning of structural missingness (e.g., students who
# never attended orientation).
#
# Key imputations:
#   1. Orientation fields (~81% missing): 
#        - Numeric counts → replaced with 0.
#        - Dates → kept as NaT (structural absence).
#   2. Academic performance & distances:
#        - Numeric columns → imputed with column median.
#   3. Demographics:
#        - Age → imputed with median.
#        - Dutch nationality → imputed with mode.
#   4. Date fields (SKC, Exam):
#        - Converted to datetime and imputed with median date.
#
# New columns created:
#   For every column imputed or structurally missing, a binary flag column
#   `<original_column>_missing_flag` is added, where:
#       1 = value was originally missing
#       0 = value was present
#
# Total new columns added: 20
#   - 5 for orientation fields
#   - 6 for academic performance
#   - 6 for distance/postcode data
#   - 2 for demographics
#   - 1 for date-type fields (SKC, exam)
#
# Output:
#   Returns a new DataFrame with all imputations applied and flags added.
# =============================================================================

def impute_all(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()

    # -----------------------------
    # 1) Define column groups
    # -----------------------------
    orientation_count_cols = [
        "sdo2_orientation_Number_of_Event_Types",
        "sdo2_orientation_Number_of_Events_Attended",
        "sdo2_orientation_STUDENTNUMMER",
    ]
    orientation_date_cols_structural = [
        "sdo2_orientation_First_Event_Date",
        "sdo2_orientation_Last_Event_Date",
    ]

    date_cols_median = [
        "sdo2_skc_SKC_DATUM",
        "sdo1_previous_Final_Exam_Date",
    ]

    academic_numeric = [
        "sdo6_results_Average_Grade_B",
        "sdo6_results_Average_Grade_A",
        "sdo6_results_Percentage_Credits_B",
        "sdo6_results_Potential_Credits_B",
        "sdo6_results_Percentage_Credits_A",
        "sdo6_results_Potential_Credits_A",
    ]

    distance_numeric = [
        "sdo1_prev_distance_postcode",
        "sdo1_prev_distance_distance_to_3584_CS",
        "sdo1_prev_distance_distance_to_3812_PA",
        "sdo1_postal_distance_postcode",
        "sdo1_postal_distance_distance_to_3812_PA",
        "sdo1_postal_distance_distance_to_3584_CS",
    ]

    demo_age_median = ["sdo1_characteristics_age_start_study"]
    demo_nat_mode   = ["sdo1_characteristics_Dutch_nationality"]

    all_date_cols = list(dict.fromkeys(orientation_date_cols_structural + date_cols_median))

    # -----------------------------
    # 2) Helpers
    # -----------------------------
    def add_flag(col: str):
        flag = f"{col}_missing_flag"
        df[flag] = df[col].isna().astype("int8")

    def to_datetime_cols(cols):
        for c in cols:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c], errors="coerce")

    def median_datetime(series: pd.Series) -> pd.Timestamp:
        return series.median()

    def mode_safe(series: pd.Series):
        m = series.mode(dropna=True)
        return m.iloc[0] if len(m) > 0 else np.nan

    def coerce_numeric(cols):
        for c in cols:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

    # -----------------------------
    # 3) Convert datatypes
    # -----------------------------
    to_datetime_cols(all_date_cols)
    coerce_numeric(
        orientation_count_cols + academic_numeric + distance_numeric +
        demo_age_median + demo_nat_mode
    )

    # -----------------------------
    # 4) Orientation (structural)
    # -----------------------------
    for c in orientation_count_cols:
        if c in df.columns:
            add_flag(c)
            df[c] = df[c].fillna(0)

    for c in orientation_date_cols_structural:
        if c in df.columns:
            add_flag(c)  # keep NaT (no imputation)

    # -----------------------------
    # 5) Date columns (partial missingness → median date)
    # -----------------------------
    for c in date_cols_median:
        if c in df.columns:
            add_flag(c)
            med = median_datetime(df[c])
            if not pd.isna(med):
                df[c] = df[c].fillna(med)

    # -----------------------------
    # 6) Numeric columns → median
    # -----------------------------
    for group in [academic_numeric, distance_numeric, demo_age_median]:
        for c in group:
            if c in df.columns:
                add_flag(c)
                med = df[c].median(skipna=True)
                df[c] = df[c].fillna(med)

    # -----------------------------
    # 7) Categorical / binary numeric → mode
    # -----------------------------
    for c in demo_nat_mode:
        if c in df.columns:
            add_flag(c)
            m = mode_safe(df[c])
            df[c] = df[c].fillna(m)

    # -----------------------------
    # 8) Clean-up for numeric consistency
    # -----------------------------
    if "sdo2_orientation_STUDENTNUMMER" in df.columns:
        df["sdo2_orientation_STUDENTNUMMER"] = df["sdo2_orientation_STUDENTNUMMER"].astype("int64")

    return df

# Example usage:
data = impute_all(data)


In [ ]:
# Confirm NaN count per column in data
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False).head(20)
print(nan_sums)

In [ ]:
data.columns

In [ ]:
data.shape

In [ ]:
data.to_csv(f"{OUT_DIR}/missing_values_imputed.csv", index=False)
print(f"✅ Saved to: {OUT_DIR}/missing_values_imputed.csv")